In [24]:
import os
from pathlib import Path
from urllib.parse import urlparse
from pydantic import BaseModel, HttpUrl
import pandas as pd
import pymupdf
import rich
from download import download
from typing import Any
from datetime import date

In [ ]:
_expected = (
    "../data/raw/has-publications-split/json/RecommandationsProfessionnelles.json"
)
if os.path.exists(_expected):
    df = pd.read_json(_expected)
    rich.print(df.head)
else:
    raise FileNotFoundError("couldn't find the file you're looking for")

In [ ]:
documentLinkSet = df["documentLinkSet"][0]
rich.print(documentLinkSet)

#### Retrieve document urls by Key (resolvedUrl)
We build a nested list where each inner list contains the `resolvedUrl` values from the dictionaries in one documentLinkSet row of `df`. We loops over every index in `df["documentLinkSet"]` and extracts `doc['resolvedUrl']` for each document.

In [ ]:
document_retrieved = [
    [doc["resolvedUrl"] for doc in df["documentLinkSet"][i]]
    for i in range(len(df["documentLinkSet"]))
]
rich.print(document_retrieved)

Sanitize the url to only get `*.pdf` files

In [ ]:
pdf_links = [
    url
    for sublist in document_retrieved
    for url in sublist
    if url.lower().split("?")[0].endswith(".pdf")
]

rich.print(pdf_links)

We download using the `download` module. (CAUTION: it will download over `1000` pdf docs)

In [ ]:
output_dir = Path("raw/")
output_dir.mkdir(parents=True, exist_ok=True)
for url in pdf_links:
    filename = Path(urlparse(url).path).name
    download(url, output_dir / filename)

### Extract title 
#### Open a pdf file 


In [4]:
doc = pymupdf.open("../data/raw/pdf/raw/_argumentaire_pec_dentaire_am_mel.pdf")

to extract the title let's first try to look for it in the metadata

In [ ]:
meta_data = doc.metadata
rich.print(meta_data)

{
    'format': 'PDF 1.7',
    'title': 'Prise en charge bucco-dentaire des patients à risque d’endocardite infectieuse',
    'author': 'MAINGUY Albane',
    'subject': '',
    'keywords': '',
    'creator': 'Microsoft® Word pour Microsoft\xa0365',
    'producer': 'Microsoft® Word pour Microsoft\xa0365',
    'creationDate': "D:20241025120102+02'00'",
    'modDate': "D:20241025120102+02'00'",
    'trapped': '',
    'encryption': None
}

`metadata` is a `dict` type, so we can get the title just by getting the `title` key.
```python
def get_title(doc): 
    doc.metadata.get("title", "").strip()
```

However, there might not be any title in the metadata, we can look for the text the largest font size in the doc. Typically we often find them in the first page of every documents so we'll assume that its the case and we will only exploit the first page of each document.

In [22]:
def get_title(doc):
    meta_data = doc.metadata.get("title", "").strip()
    if meta_data:
        return meta_data
    page = doc[0]  # this a Page object
    blocks = page.get_text("dict")["blocks"]  #  May include text and images.
    max_size = 0
    title = ""
    for block in blocks:
        if block["type"] == 0 or "line" in block:
            for line in block["lines"]:
                for span in line["spans"]:
                    if span["size"] > max_size or span["size"] == max_size:
                        # print(f"found new max_size {max_size} ")
                        max_size = span["size"]
                        title += span["text"]
    return title.strip()

In [26]:
page = doc[0]  # this a Page object
blocks = page.get_text("dict")["blocks"] 
rich.print(blocks)

[
    {
        'type': 1,
        'number': 0,
        'bbox': (224.8000030517578, 56.749961853027344, 370.5, 113.39996337890625),
        'width': 343,
        'height': 133,
        'ext': 'png',
        'colorspace': 3,
        'xres': 96,
        'yres': 96,
        'bpc': 8,
        'transform': (145.6999969482422, 0.0, -0.0, 56.650001525878906, 224.8000030517578, 56.749961853027344),
        'size': 8839,
        'image': 
b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01W\x00\x00\x00\x85\x08\x06\x00\x00\x00\xef\xb0\xbb\x88\x00\x00\x00\
tpHYs\x00\x00\x0e\xc4\x00\x00\x0e\xc4\x01\x95+\x0e\x1b\x00\x00"9IDATx\x9c\xed]\xdb\x8b\\\xc9yo\x87\x90\x90\xe0\xe0x
u\x19I\xab\xd5H\xa3\xb5\x13\x0b\xdbk\xac\xecz\xa4\x8ct\xdc\xb7\xd3\xed\xccC\x9e\xe6!\x81d!\x84}X\x13\xc2\xb2\x98\xd
8\x810`\xdd\xe6\xd2\x97\x11\x81%zN 
\xb0\xaf\x01\x87<9\x90\xe0\x97\x18c\x82\xb17\xbb^igF\xeb\xbc\xe5_\xa8\x9c\xafN}\xe7T\x9f\xae\xdb\xe9[\x9d:\xfa\x1e~
\x9a\xd1t\xf7\xe9\xafn\xbf\xfan\xf5U\xa3\xd1h0\x02\x81@ ,\x1c\xde\x05 
\x10\x08\x84:\xc2\xbb\x00\x04B9\xdc\x19\xb0F\xf4\x905\x9a\x07\xc9\xcf\xdd\xe4o\xcc\xbfL\x04\xc24\xbc\x0b@ 
\xb8\xa3\xb3\xc7\x1a\xfd1k|\xeb(\xfd\xd9\x1f\xb1\xc6\xf6{\xfe\xe5"\x10\xa6\xe1]\x00\x02\xc1\x1d=A\xac\x08 
\xd8\xce\xc8\xbf\\\x04\xc24\xbc\x0b@ 
\xb8a\xfbQB\xae\xa3Ir\x05\xc4C\xff\xb2\x11\x08\xd3\xf0.\x00\x81`\xc7\xce\xfb\t\xb1\x0e\xd4\xe4\n\xdak<\xf0/#\x810\t
\xef\x02\x10\x08v4\xef%\x04:\x9e&V$\xd7\x1ei\xaf\x84\xca\xc1\xbb\x00\x04\x82\x1d:\xad\x15\x01\xafA\xf6\x80o9\t\x84\
x1c\xde\x05 
\x10,\x88\xd2\xac\x00\x1d\xb1\xa2\xf6\xdaM\xb4WFiY\x84\xca\xc0\xbb\x00\x04\x82\x19@\x9a\xc5,\x01\x1d\xc1\xbe\xb1\xe
b_\xdeU\x01\xfc\xd0\xb7\xf7Y\xe3V\xd2\xe6;\xf7\x12\x1c\xa6\xda;\x00\x82\x7f\xcd\xc34\x0f\x18~\xbf\xf9\xc4\xbf\xbc/\
x1e\xbc\x0b@ 
\xe8\xf1\xfa\xc3\xd4\xe4\xef\xbb\x90\xeb(}\xafo\x99\x97\x85\x9d\x9d4\xcf\xb7s\x98\xb63\x16\x9b\x0e\xf8\x9b{\xa3I\xf
4\xc5k\xe0\xa7\x86\xf7\xc1Op\xadt\x13\xe2m%\xb8\xf9\x96\xff\xf6\xd4\x1f\xde\x05\x08\x1b0\x81\xfb\x8f\xd5\x80I\xde\x
adH\xa0\x85/8\x8d\x9c\x00X|\xbeeT\xa1;2\xfbZ\xa7|\xaf\xe3\xfahi\xd0\x8e\xad\xef\xb1F\xfb0\xdd8\xfaE\x02-\xd1/\xf2\x
e6\xc3\tY\x90s?!\xdc\xad\x07\xfe\xdbZOx\x17 
l\x80\xd6`2S\xab\xa2I\x99\xe4\xc4\x80\x90o\x19\x8b\xd8|\xc7\xcd\x1d0u\xa8`\xcf\xbf\xec\xf3\xa2\xfdH\x90\xe1\xb8|\x1
f\x94\x01Z\x05\x90\xca\x06\xee\x03\xdf\xed\xae\x17\xbc\x0b\x106L\xa4\xc5\xb5\x83\x8ah\x846\xed\xaf\x8a\xe4\xda|4\x8
3v&640\xa1}\xcb?\x0b\xc0G\ns\xaa\xef\xa0\xb1c[\xd1-\xc0\xcd\x7f\xc0\xc8\xed\xf3\xaagu\x06\xf5\xd1\xfc\xfd\xc3\xbb\x
00a#6i\xae\xe0\x16\xa8Hr\xbbu\xa1V\xc4}\x81\x00r\xd4\xe5\xb5:\x99\xbf\x81i\xaf\\K\x07\xf3\xffH\xbf\xa1\xf0Z\nB\x93\
xed$\xe3u\xe3\xed4\xa8\xa5\xeb\xbf\xc6nZ\xe0&#\xe1\x91]\x0b\x06\x17Q\xfb\xef\xfd\xf7G=\xe0]\x80\xb0a"\xd7*\xf9\\Mrr
Y\x0f\xfd\xcb(\xa3;\x879\x1c\xda\xa1\x82\xdb\xdf\x11>UM{\xd1G\xba\xfd8\xad\x086\xcbw\xf0@\xd8@\xc4\x08,\xf9\xc2\xbe
\xfb\xa3>\xf0.@\xd8\xb0j\xae\x15!-\x08\\\x98\x08\xa9[\xa1E\xc5\xb58\xc7\x0c\x01-I\x04\x12\xd8Z\xdf\xcd\x83U\xbav\x8
0\x99\xdf~\xa8\xd7R\xcb\x00\xdc\x0e|.\x8c\xa7\xfb\x17\xfa\x1c4]\xdf}R\x1fx\x17 
l\xd8|\xaeU!W\x9b\x16\xd8\xa9\x90\xa6\xa7\xf3\xb5\xf6\x15\x84`\xd2^\xab^o\xe0\xf2;\xe9\xa6\xa6%Va\xc6\xc3!\x8aE\x7f
wt8\xa9\xc5\xa2\x8f\x1666\xdf\xfdR\x1fx\x17 lt\r&\x16\xaf5Z\x11\x8d\xd0\x1a\xdc\xa8\x10\xb9\xeaLW\xf4\x1b:\x91\xab 
\xa6\xa8\xc2\x87\n\x9a\x87\xfa\xf6\xa0+\xe0\xd6\x127\xe7\x16\x12,\xa6fU|3\n\x0f\xde\x05\x08\x1b]\x8b\xe6Z\x15\x8d0\
xb6\xb9\x05*\xa2a\xdf\x19\xa5A\x1dU_\xf2H\xb8\xa5\x1dJ\xcb\xa1\x82Gb\xe1$\x99i\xa3\x80\xcda\x15)e\xbc\xf8\xb8\xd8\x
b4^\xa4\xd3m\xab\x81w\x01\xc2\x86-\xa0\xd5\xaaH1\x11\xd3&\x00\xa8\x82\x86\x07u\x01@N\x95\xe9\xcf\xc9U\x8a~\x97!\xd8
W\xff\xca\x7f\xdb\x8a\xb0\x1d\x02\x80\xd7V\xb5)\xc0\x89/*8\xbe\x0cx\x17 
l\x98\xdc\x02\xb0\x80\xee\xee\xfb\x97\x11\x10[L\xe8*h-\xd1CM\xbdV)\xeb\x02\x88\xb2\xcc\xe9$0{\xabV-\x0b\x02m\xb64\x
b3\x95Z\x12Q\x1aX\xf3\xdd/\xf5\x83w\x01\xc2\x86\xc9L\xe5A\x95{\xfee\xb4\xc9\t\x04\x04\xe9@^ed\xe9\xd9w\x95V\nn\x82M
i\xf1o\x97L\x90\x8f+\xa6\x95\xf5\x0f,ZkMN\x99\x11\xbc\x0b\x106\x8c\xa4\xb5\xe4\x80

### Encoding to JSON Objects using `Pydantic`

The schema that we want to use to validate JSON objects is `{doc_id, source, title, sections[], lang, pub_date, url, doc_type, metadata}`. The first question that came to mind was: **"Why not just use a Python `dict`?"** It turns out that while a `dict` is perfectly adequate for storing arbitrary data, it provides no guarantees about the structure or types of that data. Missing fields, misspelled keys, or incorrect types are only discovered when the program attempts to use them, often resulting in runtime errors that are difficult to trace.

`Pydantic` helps with this problem by making a plan as a Python class. When you put data into this model it checks the data to make sure it is correct. The model needs certain fields to be there the values need to be the type and it tells you clearly if something is wrong. If the data is not correct Pydantic tells you what is wrong. Pydantic also makes it easy to change Python objects into JSON. This is really useful because you can use Pydantic to make sure your data is good and then easily use reuse it anywhere.
We can also use `msgspec` for a light weight and faster experience but We are not doing it for critical systems. Maybe in the future. 



In [ ]:
class Section (BaseModel) : 
    title: str
    text: str 
    grade: str | None = None 

In [ ]:
class Document(BaseModel):
    doc_id: int
    source: str | None 
    title: str | None
    sections: list[Section] = []
    lang: str | None = None 
    pub_date: date | None
    url : HttpUrl
    doc_type: str  
    metadata: dict[str, Any] = {}